# Vehicle Detection, Tracking and Turning Movement Counts

Counts **Left / Through / Right** movements for each of the **4 legs** (4 x 3 = 12 movements), separately for every category (person, bicycle, car, motorcycle, bus, truck).

**How it works**
1. You click the two ends of a red line on each leg (8 clicks in total).
2. When an object crosses a leg line going **into** the junction, that leg is recorded as its **entry** leg.
3. When the same object crosses a leg line going **out of** the junction, that leg is its **exit** leg, and the movement (Left / Through / Right) is worked out from entry leg and exit leg.

**Important:** click the legs in **clockwise order as you see them on the screen** (for example top leg, right leg, bottom leg, left leg). Left / Through / Right is worked out from this order.

Imports

In [1]:
import csv

In [2]:
import json

In [3]:
import os

In [4]:
from collections import Counter, defaultdict

In [5]:
import cv2
from ultralytics import YOLO

Settings (change things here only)

In [6]:
VIDEO_PATH = r"L:\CCTV\20250616\N02174326.mp4"

MODEL_NAME = "best.pt"     # "yolo11l.pt" is more accurate but much slower on CPU

# Names of the 4 legs, in CLOCKWISE order as seen on the screen.
# This is also the order in which you will click the lines.
LEG_NAMES = ["Leg 1", "Leg 2", "Leg 3", "Leg 4"]      # e.g. ["North", "East", "South", "West"]

# Categories to count (COCO class ids): person, bicycle, car, motorcycle, bus, truck
TRACK_CLASSES = [0, 1, 2, 3, 5, 7]

REF_POINT = "bottom"          # "bottom" = bottom-centre of the box (where it touches the road), "center" = box centre
CONF = 0.3                    # minimum detection confidence
IMGSZ = 640                   # YOLO input size (smaller = faster)
FRAME_SKIP = 1                # 1 = use every frame, 2 = every 2nd frame (faster)
UTURN_MIN_FRAMES = 30         # leaving by the same leg sooner than this is treated as jitter, not a U-turn

LINES_FILE = "lines.json"     # your clicked lines are saved here
REDRAW_LINES = False          # set True to click the 4 lines again
PICK_FRAME_INDEX = 0          # which video frame is shown for clicking the lines
CENTER_OVERRIDE = None        # normally automatic; or set e.g. (850, 400) in frame pixels

MAX_SHOW_W, MAX_SHOW_H = 1280, 720     # only the display is shrunk to this size, counting uses the full frame
WINDOW = "YOLO Object Tracking and Counting"

Load the YOLO model

In [7]:
model = YOLO(MODEL_NAME)
class_list = model.names
{c: class_list[c] for c in TRACK_CLASSES}

{0: 'motorcycle',
 1: 'bicycle',
 2: 'person',
 3: 'car',
 5: 'tractor',
 7: 'truck'}

Helper functions (run once)

In [8]:
FONT = cv2.FONT_HERSHEY_SIMPLEX
MOVES = ["Left", "Through", "Right", "U-turn"]
assert len(LEG_NAMES) == 4, "This notebook is written for a 4-leg junction"


def display_scale(frame):
    # shrink big frames so the window fits on the screen (only used for showing / clicking)
    h, w = frame.shape[:2]
    return min(1.0, MAX_SHOW_W / w, MAX_SHOW_H / h)


def put_text(img, text, org, color=(255, 255, 255), size=0.7):
    # text with a black outline so it is readable on any background
    t = max(1, round(size * 3))
    cv2.putText(img, text, org, FONT, size, (0, 0, 0), t + 2)
    cv2.putText(img, text, org, FONT, size, color, t)


def side(a, b, p):
    # the sign tells on which side of the line a->b the point p lies
    return (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])


def crossed(a, b, p0, p1):
    # True if the path p0 -> p1 crosses the line segment a-b
    if (side(a, b, p0) > 0) == (side(a, b, p1) > 0):
        return False
    return (side(p0, p1, a) > 0) != (side(p0, p1, b) > 0)


def is_inside(a, b, p, center):
    # True if point p is on the same side of line a-b as the junction centre
    return (side(a, b, p) > 0) == (side(a, b, center) > 0)


def movement(entry_leg, exit_leg):
    # legs are numbered clockwise: the next leg clockwise is on the driver's LEFT
    return {0: "U-turn", 1: "Left", 2: "Through", 3: "Right"}[(exit_leg - entry_leg) % 4]


def junction_centre(lines):
    xs = [(a[0] + b[0]) / 2 for a, b in lines]
    ys = [(a[1] + b[1]) / 2 for a, b in lines]
    return (sum(xs) / 4, sum(ys) / 4)


def pick_lines(frame):
    # shows the frame, you click START and END of each leg line, ENTER to accept
    scale = display_scale(frame)
    base = cv2.resize(frame, None, fx=scale, fy=scale) if scale < 1 else frame.copy()
    win = "Click the two ends of each leg line"
    clicks = []

    def redraw():
        img = base.copy()
        for i, p in enumerate(clicks):
            cv2.circle(img, p, 5, (0, 0, 255), -1)
            if i % 2 == 1:
                q = clicks[i - 1]
                cv2.line(img, q, p, (0, 0, 255), 3)
                put_text(img, LEG_NAMES[i // 2], ((q[0] + p[0]) // 2, (q[1] + p[1]) // 2 - 10), (0, 255, 255))
        n = len(clicks)
        if n == 8:
            c = junction_centre([(clicks[2 * i], clicks[2 * i + 1]) for i in range(4)])
            cv2.circle(img, (int(c[0]), int(c[1])), 8, (255, 0, 0), -1)
            put_text(img, "junction centre", (int(c[0]) + 12, int(c[1])), (255, 200, 0))
            msg = "ENTER = accept   |   Z = undo last click"
        else:
            end = "START" if n % 2 == 0 else "END"
            msg = f"Click the {end} of the {LEG_NAMES[n // 2]} line   |   Z = undo   |   ESC = cancel"
        put_text(img, msg, (10, 30))
        cv2.imshow(win, img)

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(clicks) < 8:
            clicks.append((x, y))
            redraw()

    cv2.namedWindow(win)
    cv2.setMouseCallback(win, on_mouse)
    redraw()

    lines = None
    while True:
        key = cv2.waitKey(30) & 0xFF
        if key == 27:                                    # ESC = cancel
            break
        if key in (ord("z"), ord("Z")) and clicks:       # Z = undo last click
            clicks.pop()
            redraw()
        if key in (13, 10) and len(clicks) == 8:         # ENTER = accept
            lines = [[[round(clicks[2 * i][0] / scale), round(clicks[2 * i][1] / scale)],
                      [round(clicks[2 * i + 1][0] / scale), round(clicks[2 * i + 1][1] / scale)]]
                     for i in range(4)]
            break
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    return lines

Draw the 4 red lines by clicking (or load the saved ones)

A window opens. For each leg click the **START** and then the **END** of the line, placing it **across the road** just before the junction. Go around the junction **clockwise**. Press **Z** to undo a click and **ENTER** to accept after the 8th click.

In [9]:
if os.path.exists(LINES_FILE) and not REDRAW_LINES:
    with open(LINES_FILE) as f:
        LINES = json.load(f)["lines"]
    print(f"Loaded the 4 lines from {LINES_FILE}  (set REDRAW_LINES = True to click them again)")
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError("Could not open the video. Check VIDEO_PATH and that the drive is connected.")
    cap.set(cv2.CAP_PROP_POS_FRAMES, PICK_FRAME_INDEX)
    ok, first_frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError("Could not read a frame from the video.")

    LINES = pick_lines(first_frame)
    if LINES is None:
        raise RuntimeError("Line selection was cancelled - run this cell again.")
    with open(LINES_FILE, "w") as f:
        json.dump({"lines": LINES}, f)
    print(f"Saved the 4 lines to {LINES_FILE}")

CENTER = CENTER_OVERRIDE or junction_centre(LINES)
for name, (a, b) in zip(LEG_NAMES, LINES):
    print(f"{name}: {a} -> {b}")
print("Junction centre:", tuple(round(v) for v in CENTER))

Loaded the 4 lines from lines.json  (set REDRAW_LINES = True to click them again)
Leg 1: [242, 111] -> [648, 36]
Leg 2: [718, 69] -> [762, 369]
Leg 3: [756, 460] -> [8, 462]
Leg 4: [20, 404] -> [186, 158]
Junction centre: (418, 259)


Run tracking and count the movements

Press **Q** with the video window selected to stop safely (or use the notebook's **Interrupt** button). The counts stay in memory, so you can still run the report cell below.

In [10]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Could not open the video. Check VIDEO_PATH and that the drive is connected.")

last_pt = {}                      # track id -> reference point in the previous frame
entry = {}                        # track id -> (entry leg, frame number)
done = set()                      # track ids whose movement is already counted
class_votes = defaultdict(Counter)    # track id -> how often each class was seen (avoids car/truck flicker)
counts = Counter()                # (category, entry leg, movement) -> number
frame_no = 0


def leg_total(leg, move):
    return sum(n for (c, lg, m), n in counts.items() if lg == leg and m == move)


try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_no += 1
        if frame_no % FRAME_SKIP:
            continue

        ui = 1 / display_scale(frame)          # keeps text / lines readable on big frames

        # Run YOLO tracking on the frame
        results = model.track(frame, persist=True, tracker="bytetrack.yaml",
                              classes=TRACK_CLASSES, conf=CONF, imgsz=IMGSZ, verbose=False)
        boxes = results[0].boxes

        # Draw the 4 red lines
        for name, (a, b) in zip(LEG_NAMES, LINES):
            cv2.line(frame, tuple(a), tuple(b), (0, 0, 255), max(2, round(3 * ui)))
            put_text(frame, name, (a[0], a[1] - 10), (0, 255, 255), 0.6 * ui)
        cv2.circle(frame, (int(CENTER[0]), int(CENTER[1])), round(6 * ui), (255, 0, 0), -1)

        # Ensure there are tracked objects in this frame
        if boxes is not None and boxes.id is not None:
            for (x1, y1, x2, y2), track_id, class_idx in zip(boxes.xyxy.int().cpu().tolist(),
                                                              boxes.id.int().cpu().tolist(),
                                                              boxes.cls.int().cpu().tolist()):
                cx = (x1 + x2) // 2
                cy = y2 if REF_POINT == "bottom" else (y1 + y2) // 2
                pt = (cx, cy)
                class_name = class_list[class_idx]
                class_votes[track_id][class_name] += 1

                # Did this object cross one of the 4 lines since the previous frame?
                prev = last_pt.get(track_id)
                if prev is not None and track_id not in done:
                    for leg, (a, b) in enumerate(LINES):
                        if not crossed(a, b, prev, pt):
                            continue
                        if is_inside(a, b, pt, CENTER):
                            # moving INTO the junction -> this is its entry leg
                            if track_id not in entry:
                                entry[track_id] = (leg, frame_no)
                        elif track_id in entry:
                            # moving OUT of the junction -> this is its exit leg
                            entry_leg, entry_frame = entry[track_id]
                            if leg == entry_leg and frame_no - entry_frame < UTURN_MIN_FRAMES:
                                continue                    # jitter around the line, not a real exit
                            category = class_votes[track_id].most_common(1)[0][0]
                            counts[(category, entry_leg, movement(entry_leg, leg))] += 1
                            done.add(track_id)
                        break
                last_pt[track_id] = pt

                cv2.circle(frame, pt, round(4 * ui), (0, 0, 255), -1)
                cv2.putText(frame, f"ID: {track_id} {class_name}", (x1, y1 - 10),
                            FONT, 0.6 * ui, (0, 255, 255), max(2, round(2 * ui)))
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), max(2, round(2 * ui)))

        # Running totals on the video (all categories together)
        y = round(30 * ui)
        for leg, name in enumerate(LEG_NAMES):
            put_text(frame, f"{name}: L {leg_total(leg, 'Left')}  T {leg_total(leg, 'Through')}  R {leg_total(leg, 'Right')}",
                     (10, y), (0, 255, 255), 0.7 * ui)
            y += round(32 * ui)

        # Show the frame (shrunk only for display)
        s = display_scale(frame)
        cv2.imshow(WINDOW, cv2.resize(frame, None, fx=s, fy=s) if s < 1 else frame)

        # Exit loop if "Q" is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

except KeyboardInterrupt:
    print("Stopped by user.")

finally:
    # Release Resources
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

print(f"Finished at frame {frame_no}. Run the next cell to see the tables.")

Finished at frame 276. Run the next cell to see the tables.


Results: 12 movements x every category (also saved to turning_counts.csv)

In [11]:
categories = [class_list[c] for c in TRACK_CLASSES]

for leg, leg_name in enumerate(LEG_NAMES):
    print(f"\n=== Entering from {leg_name} ===")
    print(f"{'Category':<12}" + "".join(f"{m:>9}" for m in MOVES) + f"{'Total':>9}")
    col = dict.fromkeys(MOVES, 0)
    for c in categories:
        row = [counts[(c, leg, m)] for m in MOVES]
        for m, n in zip(MOVES, row):
            col[m] += n
        print(f"{c:<12}" + "".join(f"{n:>9}" for n in row) + f"{sum(row):>9}")
    print(f"{'ALL':<12}" + "".join(f"{col[m]:>9}" for m in MOVES) + f"{sum(col.values()):>9}")

with open("turning_counts.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["entry_leg", "movement", "category", "count"])
    for leg, leg_name in enumerate(LEG_NAMES):
        for m in MOVES:
            for c in categories:
                w.writerow([leg_name, m, c, counts[(c, leg, m)]])
print("\nSaved turning_counts.csv")


=== Entering from Leg 1 ===
Category         Left  Through    Right   U-turn    Total
motorcycle          0        2        0        0        2
bicycle             0        0        0        0        0
person              0        0        0        0        0
car                 0        0        0        0        0
tractor             0        0        0        0        0
truck               0        0        0        0        0
ALL                 0        2        0        0        2

=== Entering from Leg 2 ===
Category         Left  Through    Right   U-turn    Total
motorcycle          0        0        2        0        2
bicycle             0        0        0        0        0
person              1        0        0        0        1
car                 0        0        0        0        0
tractor             0        0        0        0        0
truck               0        0        0        0        0
ALL                 1        0        2        0        3

=== Entering 